# DESI x ACT kSZ HDF5 Prep

This notebook documents the transfer products for measuring velocity-aware DESI x ACT kSZ angular spectra. It wraps `scripts/prepare_act_desi_ksz_hdf5.py`, visualizes the compact HDF5 outputs, and leaves a concrete NaMaster starting point for the later analysis machine.

Reference paper on disk: `/global/cfs/cdirs/lsst/www/shivamp/DESI/2604.19744v1.pdf`.

Velocity convention saved here: `vr_over_c = source column 15 / 3.0e5`; no sign flip is applied.

In [ ]:
from pathlib import Path
import json
import numpy as np
import h5py
import matplotlib.pyplot as plt

def find_transfer_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'manifest.json').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the act_desi_ksz_transfer package.')


OUTDIR = find_transfer_root()
CAT_ALL = OUTDIR / 'data/desi_dr10_extended_velocity_catalogs/desi_dr10_extended_all_pz_compact.h5'
ACT_H5 = OUTDIR / 'data/act_dr6_cmb_temperature/act_dr6_hilc_fullres_tt_17000_mask_transfer.h5'
TSZ_H5 = OUTDIR / 'data/act_dr6_tsz_compton_y/act_dr6_nilc_compton_y_deproj_cib_cibdbeta_1p7_10p7_transfer.h5'
LENSING_H5 = OUTDIR / 'data/act_dr6_lensing_kappa/act_dr6_lensing_v1_baseline_kappa_transfer.h5'
HP_H5 = OUTDIR / 'data/desi_dr10_healpix_quicklooks/desi_healpix_nside512_quicklook.h5'
MANIFEST = OUTDIR / 'manifest.json'

print(OUTDIR)
if MANIFEST.exists():
    print(json.dumps(json.loads(MANIFEST.read_text()), indent=2)[:3000])

## Optional: regenerate the products

The script loads each full ASCII catalog with `np.loadtxt`, keeps only the relevant columns, then writes the ACT map and mask from pixell FITS into HDF5. Regeneration can take a while because the ASCII catalogs and ACT maps are large.

In [ ]:
# Uncomment to regenerate everything on a machine with the required inputs mounted.
# %run ../scripts/prepare_act_desi_ksz_hdf5.py --force

## Inspect HDF5 schemas

In [ ]:
def show_h5(path):
    print(path)
    with h5py.File(path, 'r') as f:
        print('attrs:')
        for key, val in f.attrs.items():
            sval = str(val)
            print(f'  {key}: {sval[:220]}')
        print('datasets:')
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f'  {name}: shape={obj.shape}, dtype={obj.dtype}')
        f.visititems(visitor)

show_h5(CAT_ALL)
show_h5(ACT_H5)
show_h5(TSZ_H5)
show_h5(LENSING_H5)
show_h5(HP_H5)

## Catalog visualizations

In [ ]:
with h5py.File(CAT_ALL, 'r') as f:
    z = f['catalog/z'][:]
    ra = f['catalog/ra_deg'][:]
    dec = f['catalog/dec_deg'][:]
    vr = f['catalog/vr_over_c'][:]
    pz = f['catalog/pz_bin'][:]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for pz_bin in (1, 2, 3, 4):
    sel = pz == pz_bin
    axes[0].hist(z[sel], bins=np.linspace(0, 2, 121), histtype='step', lw=1.5, label=f'pz{pz_bin}')
axes[0].set_xlabel('z')
axes[0].set_ylabel('N')
axes[0].legend()

axes[1].hist(vr[np.isfinite(vr)], bins=120, histtype='step', color='black')
axes[1].set_xlabel('v_los / c')
axes[1].set_ylabel('N')

h = axes[2].hist2d(ra, dec, bins=(720, 240), cmap='magma', cmin=1)
axes[2].set_xlabel('RA [deg]')
axes[2].set_ylabel('Dec [deg]')
fig.colorbar(h[3], ax=axes[2], label='objects / bin')
fig.tight_layout()

## ACT quicklook arrays

The HDF5 file stores stride-32 quicklook arrays so the map and mask can be inspected without loading the full `(10320, 43200)` arrays.

In [ ]:
with h5py.File(ACT_H5, 'r') as f:
    cmb_q = f['quicklook/cmb_stride32'][:]
    mask_q = f['quicklook/mask_stride32'][:]
    print('Full shape:', f.attrs['map_shape'])
    print('Quicklook shape:', cmb_q.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, img, title, cmap in zip(axes, [cmb_q, mask_q], ['ACT CMB stride-32', 'ACT mask stride-32'], ['coolwarm', 'viridis']):
    finite = np.isfinite(img)
    vmin, vmax = np.nanpercentile(img[finite], [2, 98]) if np.any(finite) else (None, None)
    im = ax.imshow(img, origin='lower', aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()

## tSZ and lensing quicklook arrays

These HDF5 files store stride quicklooks so the Compton-y and kappa products can be checked without loading full maps. For DESI galaxy x tSZ or DESI galaxy x lensing, use a count/overdensity galaxy field by default; the velocity-weighted field is mainly for kSZ.

In [ ]:
with h5py.File(TSZ_H5, 'r') as f:
    y_q = f['quicklook/compton_y_stride32'][:]
    y_mask_q = f['quicklook/footprint_mask_stride32'][:]
    print('tSZ full shape:', f.attrs['map_shape'])

with h5py.File(LENSING_H5, 'r') as f:
    kappa_q = f['quicklook/kappa_stride16'][:]
    kappa_mask_q = f['quicklook/mask_stride16'][:]
    print('lensing full shape:', f.attrs['map_shape'])

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
items = [
    (y_q, 'ACT NILC Compton-y', 'coolwarm'),
    (y_mask_q, 'NILC footprint mask', 'viridis'),
    (kappa_q, 'ACT DR6 lensing kappa', 'coolwarm'),
    (kappa_mask_q, 'ACT DR6 lensing mask', 'viridis'),
]
for ax, (img, title, cmap) in zip(axes.ravel(), items):
    finite = np.isfinite(img)
    vmin, vmax = np.nanpercentile(img[finite], [2, 98]) if np.any(finite) else (None, None)
    im = ax.imshow(img, origin='lower', aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()

## HEALPix galaxy maps

This cell builds count and velocity-weighted galaxy maps directly from the catalog. For kSZ, the velocity-weighted map is the important ingredient: it stores `sum_i(v_los/c)` per pixel, then normalizes by the mean number density and subtracts the masked mean. Adjust the sign here if the final paper/theory convention requires `-v/c`.

In [ ]:
import healpy as hp

nside = 512
npix = hp.nside2npix(nside)
theta = np.radians(90.0 - dec)
phi = np.radians(ra)
pix = hp.ang2pix(nside, theta, phi)

counts_hp = np.bincount(pix, minlength=npix).astype(float)
vsum_hp = np.bincount(pix, weights=vr, minlength=npix).astype(float)

observed = counts_hp > 0
mean_count = counts_hp[observed].mean()
delta_g_hp = counts_hp / mean_count - 1.0
vg_hp = vsum_hp / mean_count
vg_hp[observed] -= np.mean(vg_hp[observed])

hp.mollview(np.where(observed, delta_g_hp, hp.UNSEEN), title='DESI count overdensity', min=-1, max=5)
hp.mollview(np.where(observed, vg_hp, hp.UNSEEN), title='DESI velocity-weighted tracer')

## Reconstruct pixell maps from HDF5

The full ACT, tSZ, and lensing arrays are stored in pixell/enmap native `(y, x)` order. Use the WCS header in `/geometry` to reconstruct an enmap. Loading full ACT/tSZ arrays can require several GB of RAM.

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from pixell import enmap

with h5py.File(ACT_H5, 'r') as f:
    header = fits.Header.fromstring(f['geometry'].attrs['map_wcs_header'], sep='\n')
    wcs = WCS(header)
    cmb_enmap = enmap.enmap(f['maps/cmb_temperature'][:], wcs)
    act_mask_enmap = enmap.enmap(f['maps/analysis_mask'][:], wcs)

print(cmb_enmap.shape, act_mask_enmap.shape, cmb_enmap.wcs)

## NaMaster pseudo-Cl skeleton

This is the later-machine measurement template. It assumes you have a HEALPix CMB map and ACT mask at the same `nside` as the galaxy maps. The conversion from pixell/enmap to HEALPix can be done with `pixell.reproject` or with the analysis pipeline's preferred map reprojection. Keep the ACT mask and any DESI footprint mask explicit in the final `mask_hp`.

In [ ]:
import pymaster as nmt

# Example placeholders. Replace with CMB and mask maps converted from the ACT enmap product.
# cmb_hp = ...          # temperature map at nside
# act_mask_hp = ...     # ACT analysis mask at nside, values in [0, 1]
# desi_mask_hp = observed.astype(float)
# mask_hp = act_mask_hp * desi_mask_hp

# Monopole treatment should be checked against the exact estimator convention.
# cmb_for_cl = cmb_hp - np.sum(mask_hp * cmb_hp) / np.sum(mask_hp)
# vg_for_cl = vg_hp - np.sum(mask_hp * vg_hp) / np.sum(mask_hp)

ell_edges = np.arange(30, 3001, 50)
bins = nmt.NmtBin.from_edges(ell_edges[:-1], ell_edges[1:])

# field_t = nmt.NmtField(mask_hp, [cmb_for_cl])
# field_vg = nmt.NmtField(mask_hp, [vg_for_cl])
# workspace = nmt.NmtWorkspace()
# workspace.compute_coupling_matrix(field_t, field_vg, bins)
# cl_coupled = nmt.compute_coupled_cell(field_t, field_vg)
# cl_decoupled = workspace.decouple_cell(cl_coupled)[0]
# ell_eff = bins.get_effective_ells()

print('NaMaster skeleton ready; fill cmb_hp and mask_hp after reprojection.')